In [1]:
import os
import json
import pandas as pd
import vertexai
from vertexai.generative_models import GenerativeModel, Part

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

In [4]:
pip install pyinaturalist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pyinaturalist]0m [cattrs]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import time
import requests
from pyinaturalist import get_observations

In [6]:
ls

Chatbot_Demo_v1.ipynb.ipynb  cv_test.ipynb  notebook_template.ipynb*


In [5]:
# create folder names from dict
animal_scrape_dict = {
  'California Sea Lion' : 'Zalophus californianus',
  'Northern Elephant Seal': 'Mirounga angustirostris',
  'Harbor Seal' : 'Phoca vitulina',
  'Common Dolphin' : 'Delphinus delphis',
  'Bottlenose Dolphin' : 'Tursiops truncatus'

}



In [13]:
for key, value in animal_scrape_dict.items():
    print(f"Key: {key}, Value: {value}")

    folder_name = f"{key} - {value}"
    print(folder_name)
    os.makedirs(folder_name, exist_ok=True)

    print("Wait for 15 seconds...")
    time.sleep(15)

    observations = get_observations(
        #taxon_id=TAXON_ID,
        q='beached',
        search_on='description', # Can be 'description', 'tags', or 'both'
        taxon_name=value,
        photos=True,     # Must have images
        quality_grade='research', # High-quality, verified data
        per_page=200,        # Number of observations to grab
        order_by='created_at',       # or 'observed_on'
        order='desc' 
    )

    # 3. Download the Images
    for obs in observations['results']:
        # Get the URL of the first photo in the observation
        first_photo = obs['photos'][0]

        # pyinaturalist provides the 'square' URL by default.
        # We'll upgrade it to 'large' (1024px) for better quality.
        img_url = first_photo['url'].replace('square', 'large')

        # Create a unique filename using the observation ID
        file_name = f"{obs['id']}.jpg"
        file_path = os.path.join(folder_name, file_name)

        # Download and save
        try:
            #print(f"Downloading {img_url}...")
            img_data = requests.get(img_url).content
            with open(file_path, 'wb') as f:
                f.write(img_data)
        except Exception as e:
            print(f"Failed to download {img_url}: {e}")

    print(f"Finished! Check the '{folder_name}' folder.\n\n")

Key: California Sea Lion, Value: Zalophus californianus
California Sea Lion - Zalophus californianus
Wait for 15 seconds...
Finished! Check the 'California Sea Lion - Zalophus californianus' folder.


Key: Northern Elephant Seal, Value: Mirounga angustirostris
Northern Elephant Seal - Mirounga angustirostris
Wait for 15 seconds...
Finished! Check the 'Northern Elephant Seal - Mirounga angustirostris' folder.


Key: Harbor Seal, Value: Phoca vitulina
Harbor Seal - Phoca vitulina
Wait for 15 seconds...
Finished! Check the 'Harbor Seal - Phoca vitulina' folder.


Key: Common Dolphin, Value: Delphinus delphis
Common Dolphin - Delphinus delphis
Wait for 15 seconds...
Finished! Check the 'Common Dolphin - Delphinus delphis' folder.


Key: Bottlenose Dolphin, Value: Tursiops truncatus
Bottlenose Dolphin - Tursiops truncatus
Wait for 15 seconds...
Finished! Check the 'Bottlenose Dolphin - Tursiops truncatus' folder.




In [17]:
os.getcwd()

'/home/jupyter'

In [2]:
PROJECT_ID = 'spring-2026-lifejacket'
LOCATION = 'us-central1'
MODEL = 'gemini-2.5-flash'

In [3]:
valid_extensions = ('.jpg', '.jpeg', '.png')
image_tasks = []

# --- Phase 1: Scan Directory Structure ---
# Walk through the current directory and all subfolders
for subdir, dirs, files in os.walk(os.getcwd()):
    # Optional: Skip hidden directories or common virtual environments
    if '.git' in subdir or '.ipynb_checkpoints' in subdir or '__pycache__' in subdir:
        continue
        
    for filename in files:
        if filename.lower().endswith(valid_extensions):
            file_path = os.path.join(subdir, filename)
            parent_folder = os.path.basename(subdir)  # The folder immediately containing the photo
            
            image_tasks.append({
                "file_path": file_path,
                "filename": filename,
                "source_folder": parent_folder 
            })

In [21]:
len(image_tasks)

833

In [4]:

vertexai.init(project=PROJECT_ID, location=LOCATION)
model = GenerativeModel(MODEL)


/opt/conda/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [5]:

EDITABLE_PROMPT = """
Analyze this image of a beached marine animal. 
Return the following information in valid JSON format:
- "animal_identification": The species you identify. Give me the top three species you think this photo is and probabilities for each of those species being correct.
- "needs_rescue": Give me severity level from 1-5 (1 = animal is doing normal activities so leave alone, to 5 = in immediate need of rescue.
- "rescue_guidance": Short actionable steps.

JSON Output:
"""


In [6]:

# ... (keep your existing vertexai.init and model setup) ...

def assess_images_by_folder(root_dir):
    valid_extensions = ('.jpg', '.jpeg', '.png')
    
    # --- Phase 1: Group files by their parent folder ---
    folder_map = {} # Key: Folder Name, Value: List of file tasks

    for subdir, dirs, files in os.walk(root_dir):
        if any(skip in subdir for skip in ['.git', '.ipynb_checkpoints', '__pycache__']):
            continue
            
        images_in_this_folder = [f for f in files if f.lower().endswith(valid_extensions)]
        
        if images_in_this_folder:
            folder_name = os.path.basename(subdir)
            folder_map[folder_name] = []
            
            for filename in images_in_this_folder:
                folder_map[folder_name].append({
                    "file_path": os.path.join(subdir, filename),
                    "filename": filename,
                    "source_folder": folder_name
                })

    if not folder_map:
        print(f"No images found in {root_dir}.")
        return

    # --- Phase 2: Assessment & Per-Folder Export ---
    for folder_name, tasks in folder_map.items():
        print(f"\n--- Processing Folder: {folder_name} ({len(tasks)} images) ---")
        folder_results = []
        
        for row in tasks:
            print(f"Assessing {row['filename']}...")
            try:
                with open(row['file_path'], "rb") as f:
                    image_bytes = f.read()
                
                mime = "image/jpeg" if row['filename'].lower().endswith(('.jpg', '.jpeg')) else "image/png"
                image_part = Part.from_data(data=image_bytes, mime_type=mime)
                
                response = model.generate_content([EDITABLE_PROMPT, image_part])
                
                clean_json = response.text.replace('```json', '').replace('```', '').strip()
                ai_data = json.loads(clean_json)
                
                folder_results.append({
                    "original_file": row['filename'],
                    "ground_truth_species": row['source_folder'],
                    "gemini_identification": ai_data.get("animal_identification"),
                    "needs_rescue": ai_data.get("needs_rescue"),
                    "rescue_guidance": ai_data.get("rescue_guidance")
                })
                
            except Exception as e:
                print(f"Error on {row['filename']}: {e}")
                folder_results.append({
                    "original_file": row['filename'],
                    "ground_truth_species": row['source_folder'],
                    "gemini_identification": "ERROR",
                    "rescue_guidance": str(e)
                })

        # --- Phase 3: Export this specific folder's CSV ---
        if folder_results:
            df = pd.DataFrame(folder_results)
            # Clean filename for the CSV (removes spaces/special chars if necessary)
            safe_name = "".join([c for c in folder_name if c.isalnum() or c in (' ', '_')]).rstrip()
            csv_filename = f"results_{safe_name}.csv"
            df.to_csv(csv_filename, index=False)
            print(f"Successfully saved: {csv_filename}")


In [8]:
assess_images_by_folder(os.getcwd())


--- Processing Folder: Common Dolphin - Delphinus delphis (73 images) ---
Assessing 36402391.jpg...
Assessing 64611415.jpg...
Assessing 72152658.jpg...
Assessing 190705365.jpg...
Assessing 262257178.jpg...
Assessing 258225573.jpg...
Assessing 194288545.jpg...
Assessing 9917968.jpg...
Assessing 271263288.jpg...
Assessing 74074486.jpg...
Assessing 194111004.jpg...
Assessing 259016834.jpg...
Assessing 37377653.jpg...
Assessing 130214956.jpg...
Assessing 66539444.jpg...
Assessing 149083050.jpg...
Assessing 16376154.jpg...
Assessing 126793531.jpg...
Assessing 90511154.jpg...
Assessing 255759798.jpg...
Assessing 328564629.jpg...
Assessing 251060870.jpg...
Assessing 156385367.jpg...
Assessing 262971532.jpg...
Assessing 256636841.jpg...
Assessing 70541596.jpg...
Assessing 69318841.jpg...
Assessing 64994243.jpg...
Assessing 346638214.jpg...
Assessing 4905297.jpg...
Assessing 126793547.jpg...
Assessing 1009571.jpg...
Assessing 33503388.jpg...
Assessing 338053338.jpg...
Assessing 325656967.jpg..

In [ ]:
results_df
